In [ ]:
!pip install helper

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import orbax.checkpoint as checkpoint
from jax.sharding import SingleDeviceSharding

import grain.python as grain
import tiktoken
import gradio as gr
from pathlib import Path

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/tinystories-narrative-classification")

print("Path to dataset files:", path)

100%|██████████| 576M/576M [00:04<00:00, 131MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/thedevastator/tinystories-narrative-classification/versions/2


In [ ]:
# Embedding Layer
class TokenAndPositionEmbedding(nnx.Module):
    def __init__(self, maxlen, vocab_size, embed_dim, *, rngs):
        self.token_emb = nnx.Embed(vocab_size, embed_dim, rngs=rngs)
        self.pos_emb = nnx.Embed(maxlen, embed_dim, rngs=rngs)

    def __call__(self, x):
        seq_len = x.shape[1]
        positions = jnp.arange(seq_len)[None, :]
        return self.token_emb(x) + self.pos_emb(positions)

In [ ]:
# Transformer Block
class TransformerBlock(nnx.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, *, rngs):
        self.attention = nnx.MultiHeadAttention(
            num_heads=num_heads,
            in_features=embed_dim,
            qkv_features=embed_dim,
            out_features=embed_dim,
            decode=False,
            rngs=rngs
        )

    def __call__(self, x, mask=None):
        attn_out = self.attention(x, mask=mask)
        # Residual connection
        x = x + attn_out
        return x

In [ ]:
# The MiniGPT Model
class MiniGPT(nnx.Module):
    def __init__(self, maxlen=128, vocab_size=50257, embed_dim=192, num_heads=6,
                 feed_forward_dim=512, num_transformer_blocks=6, *, rngs):
        self.maxlen = maxlen

        # Initialize the layers we defined in the previous cell
        self.embedding = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim, rngs=rngs)
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, feed_forward_dim, rngs=rngs)
            for _ in range(num_transformer_blocks)
        ]
        self.output_layer = nnx.Linear(embed_dim, vocab_size, use_bias=False, rngs=rngs)

    def causal_attention_mask(self, seq_len):
        # Creates the triangular mask so the model can't "look ahead" at future words
        return jnp.tril(jnp.ones((seq_len, seq_len)))

    def __call__(self, token_ids):
        seq_len = token_ids.shape[1]
        mask = self.causal_attention_mask(seq_len)

        # Pass data through the pipeline
        x = self.embedding(token_ids)
        for block in self.transformer_blocks:
            x = block(x, mask=mask)

        logits = self.output_layer(x)
        return logits

In [ ]:
import os
import pandas as pd

# Custom Helper
def load_kaggle_stories(dataset_path, max_stories=None):
    """Finds and loads stories from the downloaded Kaggle dataset."""
    # Look for CSV files in the downloaded Kaggle path
    csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {dataset_path}")

    # Load the first available CSV (usually train.csv or similar)
    file_path = os.path.join(dataset_path, csv_files[0])
    print(f"Loading data from: {file_path}")

    df = pd.read_csv(file_path)

    # Intelligently find the column containing the stories
    text_col = next((col for col in df.columns if col.lower() in ['text', 'story', 'content']), None)

    if not text_col:
        raise ValueError(f"Could not find a text column in {file_path}. Available columns: {df.columns}")

    # Extract as a list of strings
    stories = df[text_col].dropna().astype(str).tolist()

    if max_stories:
        return stories[:max_stories]
    return stories

def generate_story(model, prompt, temperature=0.8, max_new_tokens=30):
    """Autoregressively generates new text tokens using the trained MiniGPT model."""
    tokenizer = tiktoken.get_encoding("gpt2")
    tokens = tokenizer.encode(prompt, allowed_special={'<|endoftext|>'})

    key = jax.random.PRNGKey(42)

    for _ in range(max_new_tokens):
        input_ids = jnp.array([tokens])
        logits = model(input_ids)
        next_token_logits = logits[0, -1, :]

        if temperature > 0:
            next_token_logits = next_token_logits / temperature
            key, subkey = jax.random.split(key)
            next_token = jax.random.categorical(subkey, next_token_logits)
        else:
            next_token = jnp.argmax(next_token_logits)

        next_token = int(next_token)
        tokens.append(next_token)

        if next_token == tokenizer.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})[0]:
            break

    return tokenizer.decode(tokens)

In [ ]:
# the pipeline
class StoryDataset:
    def __init__(self, stories, maxlen, tokenizer):
        self.stories = stories
        self.maxlen = maxlen
        self.tokenizer = tokenizer
        # We need the end token to pad shorter stories so all arrays are the same size
        self.end_token = tokenizer.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})[0]

    def __len__(self):
        return len(self.stories)

    def __getitem__(self, idx):
        story = self.stories[idx]
        tokens = self.tokenizer.encode(story, allowed_special={'<|endoftext|>'})

        # Truncate if it's too long
        if len(tokens) > self.maxlen:
            tokens = tokens[:self.maxlen]

        # Pad with zeros if it's too short
        tokens.extend([0] * (self.maxlen - len(tokens)))
        return tokens

def create_dataloader(stories, tokenizer, maxlen, batch_size, shuffle=False, num_epochs=1, seed=42, worker_count=0):
    """Creates a highly efficient data iterator using Grain."""
    dataset = StoryDataset(stories, maxlen, tokenizer)
    estimated_batches = len(dataset) // batch_size

    sampler = grain.IndexSampler(
        num_records=len(dataset),
        shuffle=shuffle,
        seed=seed,
        shard_options=grain.NoSharding(),
        num_epochs=num_epochs
    )

    dataloader = grain.DataLoader(
        data_source=dataset,
        sampler=sampler,
        operations=[
            grain.Batch(batch_size=batch_size, drop_remainder=True)
        ],
        worker_count=worker_count
    )
    return dataloader, estimated_batches

In [ ]:
def loss_fn(model, batch):
    inputs, targets = batch
    logits = model(inputs)
    # Using optax to calculate the cross-entropy loss
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits, targets
    ).mean()
    return loss, logits

@nnx.jit
def train_step(model, optimizer, metrics, batch):
    """
    JIT-compiled step. This runs entirely on the GPU/TPU for maximum speed.
    """
    # Calculate gradients
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch)

    # Log metrics
    metrics.update(loss=loss, logits=logits, labels=batch[1])

    # NEW FLAX 0.11 SYNTAX: We must pass 'model' to the update function
    optimizer.update(model, grads)

In [ ]:
# The training loop
import kagglehub

In [ ]:
# 1. Download and Load Data
print("Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("thedevastator/tinystories-narrative-classification")
print(f"Dataset path: {dataset_path}")

# Load a subset for quick testing (change max_stories to None for full training)
stories = load_kaggle_stories(dataset_path, max_stories=100)


Using Colab cache for faster access to the 'tinystories-narrative-classification' dataset.
Dataset path: /kaggle/input/tinystories-narrative-classification
Loading data from: /kaggle/input/tinystories-narrative-classification/validation.csv


In [ ]:
# 2. Hyperparameters & Initialization
maxlen = 128
batch_size = 32
num_epochs = 3
tokenizer = tiktoken.get_encoding("gpt2")

print("\nSetting up Data Loader...")
dataloader, batches_per_epoch = create_dataloader(
    stories, tokenizer, maxlen, batch_size, shuffle=False, num_epochs=1
)

print("Initializing Model and Optimizer...")
model = MiniGPT(vocab_size=tokenizer.n_vocab, rngs=nnx.Rngs(0))

total_steps = batches_per_epoch * num_epochs
warmup_steps = max(1, total_steps // 10)

lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=3e-4, warmup_steps=warmup_steps, decay_steps=total_steps, end_value=1e-5
)

# NEW FLAX 0.11 SYNTAX: We must specify `wrt=nnx.Param` so it knows to optimize the parameters
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=lr_schedule, weight_decay=0.01), wrt=nnx.Param)
metrics = nnx.MultiMetric(loss=nnx.metrics.Average('loss'))

# Function to shift the inputs by one position to create the targets (predicting the *next* word)
prep_target_batch = jax.vmap(lambda tokens: jnp.concatenate((tokens[1:], jnp.array([0]))))
metrics_history = {'train_loss': []}


Setting up Data Loader...
Initializing Model and Optimizer...


In [ ]:
# 3. The Training Loop
print(f"\nStarting training for {num_epochs} epochs...")
for epoch in range(num_epochs):
    step = 0
    for batch in dataloader:
        # Convert batch lists to JAX arrays
        input_batch = jnp.array(jnp.array(batch).T).astype(jnp.int32)
        target_batch = prep_target_batch(jnp.array(jnp.array(batch).T)).astype(jnp.int32)

        # Execute the JIT-compiled train step
        train_step(model, optimizer, metrics, (input_batch, target_batch))
        print(".", end="", flush=True)

        # Log metrics every 2 steps
        if (step + 1) % 2 == 0:
            for metric, value in metrics.compute().items():
                metrics_history[f'train_{metric}'].append(value)
            metrics.reset()
            current_lr = lr_schedule(step)
            print(f"\nEpoch: {epoch + 1}, Step {step + 1}, Loss: {metrics_history['train_loss'][-1]:.4f}")
        step += 1



Starting training for 3 epochs...
..
Epoch: 1, Step 2, Loss: 10.8816
...
Epoch: 2, Step 2, Loss: 10.5708
...
Epoch: 3, Step 2, Loss: 10.1906
.

In [ ]:
# 4. Save Checkpoint
checkpoint_path = Path.cwd() / "model_checkpoint.orbax"
checkpointer = checkpoint.PyTreeCheckpointer()
checkpointer.save(checkpoint_path, nnx.state(model), force=True)
print(f"\nTraining Complete! Model securely saved to {checkpoint_path}")


Training Complete! Model securely saved to /content/model_checkpoint.orbax


In [ ]:
# Inference & Gradio Chat Interface

In [ ]:
# Restore the trained model state
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

checkpoint_path = Path.cwd() / "model_checkpoint.orbax"
checkpointer = checkpoint.PyTreeCheckpointer()

try:
    restored_state = checkpointer.restore(checkpoint_path, item=nnx.state(model), restore_args=restore_args)
    nnx.update(model, restored_state)
    print(" Checkpoint loaded! Ready for inference.")
except Exception as e:
    print(f" Failed to load checkpoint. Ensure the training cell finished successfully. Error: {e}")


 Checkpoint loaded! Ready for inference.


In [ ]:
# Setup and Launch Gradio
def gradio_chat_wrapper(story_prompt, temperature, max_new_tokens):
    """Wraps our custom text generator for the Gradio UI."""
    return generate_story(model, story_prompt, temperature, max_new_tokens)

demo = gr.Interface(
    fn=gradio_chat_wrapper,
    inputs=[
        gr.Textbox(label="Story Prompt", placeholder="Start typing a story here...", value="Once upon a time a big bear "),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.8, step=0.01, label="Temperature (Creativity)"),
        gr.Slider(minimum=5, maximum=200, value=30, step=1, label="Max Tokens to Generate")
    ],
    outputs=[gr.Textbox(label="Generated Story")],
    title="TinyStories MiniGPT",
    description="A lightweight LLM built from scratch using JAX, Flax NNX, and Grain.",
    theme=gr.themes.Soft()
)

print("\nLaunching Chat Interface...")
demo.launch(share=True)


Launching Chat Interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8ff46061855f240553.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
